In [ ]:
import os
import json
import pickle

import pandas as pd
import numpy as np

from joblib import parallel_backend

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [ ]:
# set the maximum number of cores to use during gridsearch tuning
max_cores = 16

In [ ]:
# set the column name of the grouping id (e.g., Ranch_Block_Year) for tuning cross-validation
tuneby_group = 'id_column'

# set the column name of the dependent variable
y_col = 'sqrt_Biomass_kg_ha'

In [ ]:
# vegetation indices - I have decided to subset these for Thunder Basin based on results of variable importance from CPER
veg_list = [
    'NDVI', 
    'DFI', 
    'NDTI', 
    'SATVI', 
    'NDII7', 
    #'SAVI', 'RDVI', 'MTVI1', 'NCI', 'NDCI', 'PSRI', 'NDWI', 'EVI', 
    'TCBI', 
    'TCGI', 
    'TCWI',
    #'BAI_126', 'BAI_136', 'BAI_146', 'BAI_236', 'BAI_246', 'BAI_346'
]

# individual bands
band_list = [
    'BLUE', 
    'GREEN', 
    'RED',
    'NIR1', 
    'SWIR1', 
    'SWIR2'
]

# create the final list of independent variables
var_names = veg_list+band_list

In [ ]:
# helper function to calculate VIP score from PLS
def vip(x, y, model):
    #https://omicsforum.ca/t/the-calculation-of-vip-score-in-pls-da-module/3312
    t = model.x_scores_
    w = model.x_rotations_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        weight = np.array([ (w[i,j] / np.linalg.norm(w[:,j]))**2 for j in range(h) ])
        vips[i] = np.sqrt(p*(s.T @ weight)/total_s).item()
    return vips

In [ ]:
# create the base model pipeline for tuning and fitting
mod_base = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('PLS', PLSRegression(n_components=1, scale=False))
    ])

# set the method of splitting during cross-validation
mod_split = LeaveOneGroupOut()

In [ ]:
# set the tuning parameters to test
# currently setup to allow up to the total number of input variables
param_grid = {
     'PLS__n_components': [int(x) for x in np.arange(1, len(var_names))]
}

# set the scoring metrics
scoring = {
        'R2': 'r2',
        'MAE': 'neg_mean_absolute_error',
    }

In [ ]:
# set whether or not to compute variable importance 
test_vip = True

# create empty dictionary for storing results
mod_dict = {}

# create empty dictionary for storing VIP scores
df_vip = pd.DataFrame()

#### NEED TO CREATE A LIST OF THE NAMES OF THE TRAINING SITES THAT CAN BE USED TO SUBSET/CONCATENATE TRAINING DATA #####
#### NOTE THAT ITEMS IN THIS LIST WILL ALSO BE USED FOR NAMING FILES AND REFERENCING MODELS IN THE DICTIONARY #####
for train_sites in ['MR', 'RR', 'CR', 'etc...']:
    #### NEED TO SETUP THE TRAINING DATAFRAME (df) USING SUBSET OR CONCATENATE OF OTHER DATA #####
    # subset/concatenate your data to create a training dataframe called df
    df = df_all[SUBSET CRITERIA].copy()
    
    # create empty dictionary to store results for individual iteration
    mod_dict[train_sites] = {}
    
    # create object of the groups for cross-validation splits
    split_groups = df[tuneby_group]
    # create object of the independent variables
    all_x = df[var_names]
    # create object of the dependent variable
    all_y = df[y_col]
    
    # create the splitter object
    cv_splitter = mod_split.split(all_x, groups=split_groups)
    
    # create the grid search object
    grid_search = GridSearchCV(estimator=mod_base,
                               param_grid=param_grid,
                               scoring=scoring, 
                               refit='R2', 
                               return_train_score=True,
                               cv=cv_splitter, 
                               n_jobs=min(sum([len(x) for x in param_grid]), max_cores),
                               verbose=0,
                              )
    
    # run in parallel
    with parallel_backend('threading'):
        # fit the grid search to the data
        grid_search.fit(all_x, all_y)
        # create a final model with the best parameters from grid search
        mod_fnl = mod_base.set_params(**grid_search.best_params_)
        # fit the final model
        mod_fnl.fit(all_x, all_y)
        # save the CV results to the output dictionary
        mod_dict[train_sites]['CV_results'] = grid_search.cv_results_
    
    #### NEED TO CHANGE THE PATH FOR WHERE TO SAVE #####
    # save the final model for later prediction
    with open(os.path.join('./path/to/directory/', 'ffar_pls_model_' + train_sites + '.pk'), 'wb') as fp:
        pickle.dump(mod_fnl, fp, protocol=pickle.HIGHEST_PROTOCOL)
    
    if test_vip:
        var_names_out = var_names
        pls_vip = vip(all_x, all_y, mod_fnl['PLS'])
        pls_coefs = abs(mod_fnl['PLS'].coef_).squeeze()
        df_vip = pd.concat([df_vip,
                            pd.DataFrame({'Train_sites': train_sites,
                                          'Variable': var_names_out,
                                          'VIP': pls_vip,
                                          'Coef': pls_coefs})])
        mod_dict[train_sites]['VIP'] = df_vip

In [ ]:
#### NEED TO CHANGE THE PATH FOR WHERE TO SAVE #####
# Write dictionary to a JSON file
with open("/path/to/directory/ffar_pls_train_results.json", "w") as file:
    json.dump(data, file)